# v13 · Is the firing signature injective? Exhaustive over every lowercase word of length ≤ 3

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
import sys, string
from itertools import product
sys.path.insert(0, os.path.join(PROJ, "VAPMIP", "engines"))
e17 = load("VAPMIP/engines/e17_hyperindexing.py", "e17")
words = ["".join(p) for n in (1, 2, 3) for p in product(string.ascii_lowercase, repeat=n)]
print(len(words), "words")
for n in (4, 8, 16, 32):
    prim = e17.primes_up_to_n(n)
    sigs = Counter()
    for w in words:
        raw = [e17.project(w, k, prim) for k in range(n)]
        norm = math.sqrt(sum(x*x for x in raw))
        v = [x / norm for x in raw]
        sigs[tuple(sorted(range(n), key=lambda k: abs(v[k])))] += 1
    print("n=%-3d distinct signatures: %6d of %d words | largest class: %d | n! = %s" % (n, len(sigs), len(words), max(sigs.values()), math.factorial(n) if n <= 16 else "%.2e" % math.factorial(n)))

18278 words
n=4   distinct signatures:      7 of 18278 words | largest class: 14776 | n! = 24
n=8   distinct signatures:      8 of 18278 words | largest class: 14702 | n! = 40320
n=16  distinct signatures:      8 of 18278 words | largest class: 14702 | n! = 20922789888000
n=32  distinct signatures:      8 of 18278 words | largest class: 14702 | n! = 2.63e+35


## Longer texts do separate
The file's experiment used sentences of 23 to 44 characters. The same test on random 25-character strings:

In [3]:
import random
rnd = random.Random(5)
long_words = ["".join(rnd.choice(string.ascii_lowercase + " ") for _ in range(25)) for _ in range(2000)]
for n in (8, 16, 32):
    prim = e17.primes_up_to_n(n)
    sg = set()
    for w in long_words:
        raw = [e17.project(w, k, prim) for k in range(n)]
        nm = math.sqrt(sum(x*x for x in raw)); v = [x / nm for x in raw]
        sg.add(tuple(sorted(range(n), key=lambda k: abs(v[k]))))
    print("n=%-3d distinct signatures among 2000 random 25-char strings: %d" % (n, len(sg)))

n=8   distinct signatures among 2000 random 25-char strings: 667
n=16  distinct signatures among 2000 random 25-char strings: 1862
n=32  distinct signatures among 2000 random 25-char strings: 1862


The number of possible signatures is n!, far above 18,278 from n = 8 on, so injectivity is not blocked by counting. What limits it is the projection: the value in dimension k is a real sum and the signature keeps only the order of magnitudes. On words of length ≤ 3 the signature takes 7–8 values in total at every n from 4 to 32, so it is far from injective there; on 25-character strings (the cell above) it separates much more. The file's 'effectively injective at N = 8' is a statement about ten long sentences, not about the map.